# Building a Training Pipeline using Pytorch from a real word data

We will take a real world dataset and build a Neural Network using PyTorch to solve a regression problem

- We will use a simple NN
- train it on a real world dataset
- will mimic the pytorch workflow and see how it works
- will have a lot of manual elements
- end results will be bad. obviously.

## Code Flow

1. Bring the data using `pandas`
2. basic preprocessing and setup
3. training process:
    - create the model
    - f/w pass
    - loss calc
    - backprop
    - params update
4. evaluation

# Part 1: Fetching the data

In [51]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [52]:
# Importing the dataset

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [53]:

# REMEMBER: when rerunning this cell, rerun the cell that imports the dataset first to get the columns back
# otherwise, you will get an error that the columns are not found because they have already been dropped

df.shape

# drop unneeded columns

df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [54]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# Part 2: Preprocessing and setup



We use `StandardScaler` to scale the data. It converts each entry into $z = \frac{x - \mu}{\sigma}$ where $\mu$ is the mean and $\sigma$ is the standard deviation. This means that the data will have a mean of 0 and a standard deviation of 1. (TODO: gotta learn the theory behind this!)

In [55]:
scaler = StandardScaler()

# first fit(learn mean and std from training data) and transform it, then transform the test data (don't fit it to the test data)
X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

X_test

array([[ 1.60042035,  0.18463838,  1.54577751, ...,  0.71366303,
        -0.55696798,  0.39102388],
       [-0.3821043 , -0.42017343, -0.43534126, ..., -1.19648454,
        -1.17945134, -1.31091562],
       [-0.51206059,  0.69370777, -0.54735928, ..., -0.78097633,
         0.3362308 , -0.11939385],
       ...,
       [-0.61160158, -0.26138114, -0.58176482, ..., -0.01114932,
         0.81768277, -0.03076826],
       [ 0.2842673 ,  2.41473596,  0.19396   , ..., -0.74435423,
         0.5518305 , -1.23432561],
       [-0.31850867, -0.80547825, -0.34892735, ..., -0.13627482,
         0.79012491,  0.63611192]], shape=(114, 30))

We then use `LabelEncoder` to convert the labels into integers. This is because the labels are currently in string format and we need them in integer format to train the model.

In [56]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)

y_test = encoder.transform(y_test)

y_test

array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0])

All of the data are `numpy` arrays now. We will convert them into `torch` tensors.

In [57]:
X_train_tensor = torch.from_numpy(X_train).type(torch.float64)
X_test_tensor = torch.from_numpy(X_test).type(torch.float64)
y_train_tensor = torch.from_numpy(y_train).type(torch.long)
y_test_tensor = torch.from_numpy(y_test).type(torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([455, 30])
torch.Size([455])


# Defining the model and training loop

In [89]:

class MySimpleNN():
    def __init__(self, X_train):
    
        # there are 30 weights to learn (one for each feature) and 1 bias to learn
        self.weights : torch.Tensor = torch.rand(X_train.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias : torch.Tensor = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        
        
    def forward(self, X_train):
        
        # z = w * X + b
        z = torch.matmul(X_train, self.weights) + self.bias
        
        y_pred = torch.sigmoid(z)
    
        return y_pred
    
    def loss(self, y_pred, y):
        
        epsilon = 1e-7  # to avoid log(0)
        
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)  # to ensure y_pred is in the range (0, 1)
        
        # since this is a binary classification problem, we will use binary cross entropy loss
        
        loss = - (y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
        
        return loss
    
    
        


## Some important params

In [94]:
# needed for gradient descent or other form of optimization
from numpy import float64


learning_rate = torch.tensor(0.1, dtype=torch.float64)
# how many runs
epochs = 250


In [95]:
# create an instance of the model

model = MySimpleNN(X_train_tensor)

print(model.weights)
print(model.bias)


# this training is done in a loop

for epoch in range(epochs):
    
    # f/w pass
    y_pred = model.forward(X_train_tensor)
    
    # print(y_pred[:5])
    

    # loss calc
    
    loss = model.loss(y_pred, y_train_tensor)
    
    # print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

    # b/w pass
    loss.backward()

    # update weights and bias
    
    # w_new = w_old - learning_rate * dw
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
        
        # print(f"Epoch: {epoch + 1}, weights: {model.weights.squeeze().tolist()}, bias: {model.bias.item()}, loss: {loss.item()}")

    # zero out the gradients for the next iteration
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    
    # print loss in each epoch
    # print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

tensor([[0.4807],
        [0.2213],
        [0.7582],
        [0.4978],
        [0.0628],
        [0.7328],
        [0.6892],
        [0.5310],
        [0.3238],
        [0.5865],
        [0.0932],
        [0.3324],
        [0.0180],
        [0.1196],
        [0.4058],
        [0.5756],
        [0.3929],
        [0.0013],
        [0.4739],
        [0.1670],
        [0.8367],
        [0.6752],
        [0.8679],
        [0.2645],
        [0.6194],
        [0.8970],
        [0.0964],
        [0.5502],
        [0.4069],
        [0.5445]], dtype=torch.float64, requires_grad=True)
tensor([0.], dtype=torch.float64, requires_grad=True)


In [96]:
print(model.weights)
print(model.bias)

tensor([[-0.0890],
        [-0.1401],
        [ 0.1645],
        [-0.0467],
        [-0.1566],
        [-0.0169],
        [ 0.1149],
        [-0.0690],
        [ 0.0420],
        [ 0.1850],
        [-0.0074],
        [ 0.0013],
        [-0.1092],
        [ 0.0177],
        [ 0.0110],
        [ 0.0224],
        [ 0.2066],
        [-0.0855],
        [ 0.0839],
        [-0.1574],
        [ 0.2547],
        [ 0.1297],
        [ 0.2665],
        [-0.2747],
        [ 0.1145],
        [ 0.1682],
        [-0.3872],
        [ 0.0005],
        [-0.1343],
        [-0.0180]], dtype=torch.float64, requires_grad=True)
tensor([-0.5288], dtype=torch.float64, requires_grad=True)


## Model Evaluation

Note that the original data for `y_train_tensor` are all in 0 or 1. So we decide a "threshold" value. e.g. 0.5 for `y_pred_test`. If the value is above 0.5, we will consider it as 1. If the value is below 0.5, we will consider it as 0. This is a common practice in binary classification problems.

In [104]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    
    y_pred = (y_pred > 0.5).type(torch.float64)
    
    # calculate accuracy
    
    accuracy = (y_pred == y_test_tensor).float().mean()
    
    print(accuracy)

tensor(0.6228)


So this was the training pipeline for a simple NN using PyTorch.